# Exercício 1 - Análise de Dados - Conectando com o Banco de Dados <img src="https://raw.githubusercontent.com/devicons/devicon/master/icons/microsoftsqlserver/microsoftsqlserver-plain.svg" height="45" />🏢

![Jupyter](https://img.shields.io/badge/Jupyter-111827?style=flat-square&logo=jupyter&logoColor=F37626)
![Python](https://img.shields.io/badge/Python-111827?style=flat-square&logo=python&logoColor=3776AB)
![pyodbc](https://img.shields.io/badge/pyodbc-0078D4?style=flat-square)
![Python Version](https://img.shields.io/badge/python-3.14+-blue)
![Tópico](https://img.shields.io/badge/tópico-exercício%20%7C%20pyodbc%20%7C%20análise-teal)
![Dificuldade](https://img.shields.io/badge/dificuldade-Intermediário-yellow)
![Pré-req](https://img.shields.io/badge/pré--req-pyodbc%20%7C%20pandas-purple)
![Biblioteca](https://img.shields.io/badge/requer-pyodbc%20%2B%20pandas-orange)

> Cenário: você acabou de entrar numa empresa que registra as vendas do ano inteiro direto num SQL Server. Ninguém vai te dar um Excel pronto — o trabalho começa **conectando no banco da empresa** e entendendo o que existe lá dentro, antes de qualquer análise.

## 📋 Conteúdo

1. [Simulando a Base da Empresa](#-1-simulando-a-base-da-empresa)
2. [Conectando no Banco](#-2-conectando-no-banco)
3. [Descobrindo as Tabelas Disponíveis](#-3-descobrindo-as-tabelas-disponíveis)
4. [Explorando a Estrutura da Tabela](#-4-explorando-a-estrutura-da-tabela)
5. [Uma Primeira Olhada nos Dados](#-5-uma-primeira-olhada-nos-dados)


## 🏗️ 1. Simulando a Base da Empresa

Pra deixar o exercício reproduzível, esta célula gera uma tabela `VendasEmpresa` com um ano inteiro de vendas fictícias — é o "banco que a empresa te entregou". Em uma situação real, essa tabela já existiria e você pularia direto pra próxima seção.

In [1]:
import random
from datetime import date, timedelta
from conexao import nova_conexao_sqlserver
from cores import *

random.seed(20)

produtos_por_categoria = {
    "Eletrônicos": ["Notebook", "Mouse sem Fio", "Teclado Mecânico", "Monitor 27\"", "Headset", "Webcam"],
    "Móveis": ["Cadeira Gamer", "Mesa de Escritório", "Estante", "Luminária de Mesa"],
    "Papelaria": ["Caderno", "Caneta", "Agenda", "Mochila"],
    "Casa": ["Liquidificador", "Cafeteira", "Panela Elétrica", "Aspirador de Pó"],
}
vendedores = ["Ana Souza", "Bruno Lima", "Carla Dias", "Diego Alves", "Elaine Rocha"]

linhas_geradas = []
data_inicial = date(2025, 1, 1)
for _ in range(300):
    categoria = random.choice(list(produtos_por_categoria.keys()))
    produto = random.choice(produtos_por_categoria[categoria])
    quantidade = random.randint(1, 10)
    valor_unitario = round(random.uniform(20, 3500), 2)
    dias_aleatorios = random.randint(0, 364)
    data_venda = data_inicial + timedelta(days=dias_aleatorios)
    vendedor = random.choice(vendedores)
    linhas_geradas.append((produto, categoria, quantidade, valor_unitario, data_venda, vendedor))

conexao = nova_conexao_sqlserver(banco="HashtagCursoSQL")
cursor = conexao.cursor()

cursor.execute("""
IF OBJECT_ID('dbo.VendasEmpresa', 'U') IS NOT NULL
    DROP TABLE dbo.VendasEmpresa;

CREATE TABLE dbo.VendasEmpresa (
    Id INT IDENTITY(1,1) PRIMARY KEY,
    Produto VARCHAR(100) NOT NULL,
    Categoria VARCHAR(50) NOT NULL,
    Quantidade INT NOT NULL,
    ValorUnitario DECIMAL(10,2) NOT NULL,
    DataVenda DATE NOT NULL,
    Vendedor VARCHAR(100) NOT NULL
);
""")
cursor.executemany(
    """INSERT INTO dbo.VendasEmpresa (Produto, Categoria, Quantidade, ValorUnitario, DataVenda, Vendedor)
       VALUES (?, ?, ?, ?, ?, ?)""",
    linhas_geradas
)
conexao.commit()
conexao.close()

print(f"{VerdeClaro}Base da empresa pronta: {len(linhas_geradas)} vendas em VendasEmpresa.{Reset}")


Base da empresa pronta: 300 vendas em VendasEmpresa.


## 🔌 2. Conectando no Banco

A partir daqui, o exercício de verdade começa: conectar e explorar, sem saber de antemão tudo que a tabela contém.

In [2]:
conexao = nova_conexao_sqlserver(banco="HashtagCursoSQL")
cursor = conexao.cursor()

print(f"{VerdeClaro}Conectado ao banco HashtagCursoSQL.{Reset}")


Conectado ao banco HashtagCursoSQL.


## 🗂️ 3. Descobrindo as Tabelas Disponíveis

Antes de escrever qualquer `SELECT`, vale perguntar pro próprio banco quais tabelas existem — `INFORMATION_SCHEMA.TABLES` é uma view padrão do SQL Server (e de outros bancos) que lista isso.

| View 🔑 | O que devolve 🔓 |
|---|---|
| `INFORMATION_SCHEMA.TABLES` | nome de todas as tabelas do banco |

In [3]:
cursor.execute("SELECT TABLE_NAME FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_TYPE = 'BASE TABLE'")
tabelas = [linha.TABLE_NAME for linha in cursor.fetchall()]

print(f"{CinzaClaro}Tabelas encontradas no banco:{Reset}")
for tabela in tabelas:
    print(f"  {VerdeClaro}{tabela}{Reset}")


Tabelas encontradas no banco:
  TesteConexao
  Vendas
  VendasEmpresa


## 🧬 4. Explorando a Estrutura da Tabela

Com `VendasEmpresa` identificada, `INFORMATION_SCHEMA.COLUMNS` mostra as colunas e os tipos — o "dicionário de dados" da tabela.

In [4]:
cursor.execute(
    """SELECT COLUMN_NAME, DATA_TYPE
       FROM INFORMATION_SCHEMA.COLUMNS
       WHERE TABLE_NAME = ?
       ORDER BY ORDINAL_POSITION""",
    "VendasEmpresa"
)
colunas = cursor.fetchall()

print(f"{CinzaClaro}Colunas de VendasEmpresa:{Reset}")
for coluna in colunas:
    print(f"  {VerdeClaro}{coluna.COLUMN_NAME}{Reset} {CinzaEscuro}({coluna.DATA_TYPE}){Reset}")


Colunas de VendasEmpresa:
  Id (int)
  Produto (varchar)
  Categoria (varchar)
  Quantidade (int)
  ValorUnitario (decimal)
  DataVenda (date)
  Vendedor (varchar)


## 👀 5. Uma Primeira Olhada nos Dados

Antes de sair analisando, vale ver uma amostra e o volume total — o equivalente a um `df.head()` direto no banco.

In [5]:
cursor.execute("SELECT COUNT(*) AS Total FROM dbo.VendasEmpresa")
total_linhas = cursor.fetchone().Total
print(f"{CinzaClaro}Total de vendas registradas:{Reset} {MagentaClaro}{total_linhas}{Reset}\n")

cursor.execute("SELECT TOP 5 * FROM dbo.VendasEmpresa ORDER BY DataVenda")
amostra = cursor.fetchall()

print(f"{CinzaClaro}Amostra (5 primeiras vendas do ano):{Reset}")
for venda in amostra:
    print(f"  {venda.DataVenda} — {VerdeClaro}{venda.Produto}{Reset} ({venda.Categoria}) — {venda.Quantidade}x R${venda.ValorUnitario}")

conexao.close()


Total de vendas registradas: 300

Amostra (5 primeiras vendas do ano):
  2025-01-03 — Caneta (Papelaria) — 8x R$2843.94
  2025-01-04 — Aspirador de Pó (Casa) — 4x R$532.78
  2025-01-06 — Webcam (Eletrônicos) — 3x R$2909.72
  2025-01-07 — Mesa de Escritório (Móveis) — 10x R$846.74
  2025-01-07 — Liquidificador (Casa) — 6x R$1815.88


A conexão está feita, a estrutura está mapeada e já se sabe o volume de dados envolvido. O próximo notebook usa exatamente esses mesmos dados pra construir a análise de verdade, com pandas.

> ▶️ Próximo notebook: **Exercício 1 - Análise de Dados - Construindo a Análise**.